In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
titanic = pd.read_csv('train.csv')
titanic

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [3]:
titanic.drop(columns = ['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)
titanic

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [4]:
X_train, X_test, y_train, y_test = train_test_split(titanic.drop(columns=['Survived']), titanic['Survived'], test_size=0.2, random_state=52)
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
694,1,male,60.0,0,0,26.5500,S
828,3,male,NaN,0,0,7.7500,Q
856,1,female,45.0,1,1,164.8667,S
119,3,female,2.0,4,2,31.2750,S
642,3,female,2.0,3,2,27.9000,S
...,...,...,...,...,...,...,...
86,3,male,16.0,1,3,34.3750,S
151,1,female,22.0,1,0,66.6000,S
525,3,male,40.5,0,0,7.7500,Q
779,1,female,43.0,0,1,211.3375,S


In [5]:
X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
694,1,male,60.0,0,0,26.5500,S
828,3,male,NaN,0,0,7.7500,Q
856,1,female,45.0,1,1,164.8667,S
119,3,female,2.0,4,2,31.2750,S
642,3,female,2.0,3,2,27.9000,S
...,...,...,...,...,...,...,...
86,3,male,16.0,1,3,34.3750,S
151,1,female,22.0,1,0,66.6000,S
525,3,male,40.5,0,0,7.7500,Q
779,1,female,43.0,0,1,211.3375,S


In [6]:
## we used [2] because its a good way to use columns by their number value instead of their name for pipelines

trf1 = ColumnTransformer([
    ('impute_age', SimpleImputer(), [2]),
    ('impute_embarked', SimpleImputer(strategy = 'most_frequent'), [6])
], remainder = 'passthrough')

In [7]:
trf2 = ColumnTransformer([
    ('ohe_sex_embarked', OneHotEncoder(sparse_output = False, handle_unknown = 'ignore'), [1,6]),
], remainder = 'passthrough')

In [8]:
trf3 = ColumnTransformer([
    ('scale', MinMaxScaler(),slice(0,10))
])

In [9]:
trf4 = SelectKBest(score_func=chi2,k=5)

In [10]:
trf5 = DecisionTreeClassifier()

In [11]:
pipe = Pipeline([
    ('trf1', trf1),
    ('trf2', trf2),    
    ('trf3', trf3),    
    ('trf4', trf4),    
    ('trf5', trf5),    
])


## OR

# pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [12]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=5,
                             score_func=<function chi2 at 0x000001C44F3C11C0>)),
                ('trf5', DecisionTreeClassifier())])

In [13]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'trf3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=5, score_func=<function chi2 at 0x000001C44F3C11C0>),
 'trf5': DecisionTreeClassifier()}

In [14]:
pipe.named_steps['trf1'].transformers_[0][1].statistics_ ## mean

array([29.71397188])

In [15]:
from sklearn import set_config
set_config(display = 'diagram')

In [16]:
y_pred = pipe.predict(X_test)

In [17]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_pred)

0.6145251396648045

In [18]:
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv = 5, scoring = 'accuracy').mean()

np.float64(0.6418595489018024)

In [21]:
params = {
    'trf5__max_depth' : [1,2,3,4,5,None]
}

In [22]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv = 5, scoring = 'accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('trf3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trf4',
                                        SelectKBest(k=5,
                                                    score_func=<function chi2 at 0x000001C44F3C11C0>)),
                                       ('trf5', DecisionTreeClassifier())]),
             param_grid={'trf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [23]:
grid.best_score_

np.float64(0.6418595489018024)

In [24]:
grid.best_params_

{'trf5__max_depth': 1}

## Testing

In [25]:
test_input = np.array([2, 'male', 31.0, 0, 0, 10.5, 'S'], dtype = object).reshape(1,7)

In [26]:
pipe.predict(test_input)

c:\DataScience\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
c:\DataScience\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


array([0])